함수 및 패키지 호출

In [2]:
import os.path
from pathlib import Path
from glob import glob

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

import matplotlib.pyplot as plt
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

import re
import emoji
from soynlp.normalizer import repeat_normalize

import re
import pandas as pd
import random
import string

import matplotlib.pyplot as plt
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, AutoConfig, DataCollatorWithPadding
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import EarlyStoppingCallback
import torch.nn as nn
from transformers import AdamW
from transformers.modeling_outputs import SequenceClassifierOutput


device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
device

c:\Users\choiyunjin\anaconda3\envs\sm\lib\site-packages\tqdm\auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda', index=0)

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average = 'macro')
    acc = accuracy_score(labels, preds)

    return {'accuracy' : acc,
            'f1' : f1,
            'precision' : precision,
            'recall' : recall}

데이터 불러오기

In [4]:
whole_df = pd.read_csv('C:/Users/choiyunjin/Desktop/test/학습데이터/whole.csv')

In [8]:
# 모델 불러오기
from transformers import ElectraModel, ElectraTokenizer, ElectraForSequenceClassification

MODEL_NAME = "beomi/KcELECTRA-base-v2022"   #"monologg/koelectra-base-v3-discriminator" "klue/roberta-base", "beomi/KcELECTRA-base-v2022"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

Train, Valid, Test 분할

In [ ]:
train_data = whole_df.sample(frac=0.6, random_state = 42)
remaining_data = whole_df.drop(train_data.index)

valid_data = remaining_data.sample(frac=0.3 / (0.3 + 0.1), random_state=42)
test_data = remaining_data.drop(valid_data.index)

In [ ]:
# 코멘트에 대한 길이 Plot으로 확인
# Calculate the length of each comment for train, valid, and test data
train_data['comment_length'] = train_data['comment'].str.len()
valid_data['comment_length'] = valid_data['comment'].str.len()
test_data['comment_length'] = test_data['comment'].str.len()

# Plot the distribution of comment lengths for train data
plt.figure(figsize=(10, 6))
sns.histplot(data=train_data, x='comment_length', bins=30, kde=True, color='blue')
plt.xlabel('Comment Length')
plt.ylabel('Frequency')
plt.title('Distribution of Comment Lengths (Train Data)')
plt.show()

# Plot the distribution of comment lengths for valid data
plt.figure(figsize=(10, 6))
sns.histplot(data=valid_data, x='comment_length', bins=30, kde=True, color='orange')
plt.xlabel('Comment Length')
plt.ylabel('Frequency')
plt.title('Distribution of Comment Lengths (Valid Data)')
plt.show()

# Plot the distribution of comment lengths for test data
plt.figure(figsize=(10, 6))
sns.histplot(data=test_data, x='comment_length', bins=30, kde=True, color='green')
plt.xlabel('Comment Length')
plt.ylabel('Frequency')
plt.title('Distribution of Comment Lengths (Test Data)')
plt.show()

In [ ]:
class CurseDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item
    
    def __len__(self):
        return len(self.labels)

토크나이징

In [ ]:
tokenized_train_sentences = tokenizer(list(train_data['comment']),
                                      return_tensors = 'pt',
                                      max_length = 512,
                                      padding = True,
                                      truncation = True,
                                      add_special_tokens = True,)

In [ ]:
tokenized_valid_sentences = tokenizer(list(valid_data['comment']),
                                      return_tensors = 'pt',
                                      max_length = 512,
                                      padding = True,
                                      truncation = True,
                                      add_special_tokens = True,)

In [ ]:
tokenized_test_sentences = tokenizer(list(test_data['comment']),
                                      return_tensors = 'pt',
                                      max_length = 512,
                                      padding = True,
                                      truncation = True,
                                      add_special_tokens = True,)

In [ ]:
from torch.utils.data import DataLoader

train_label = train_data['label'].values
valid_label = valid_data['label'].values
test_label = test_data['label'].values

train_dataset = CurseDataset(tokenized_train_sentences, train_label)
valid_dataset = CurseDataset(tokenized_valid_sentences, valid_label)
test_dataset = CurseDataset(tokenized_test_sentences, test_label)

모델 학습

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels = 3, problem_type = 'multi-label-classification', pad_token_id = 3)  

In [ ]:
model.to(device)

In [ ]:
from transformers import Trainer
import torch.nn as nn

# 손실함수 정의
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        # forward pass
        outputs = model(**inputs) 
        logits = outputs.get("logits")

        #class_weights = torch.tensor([2.0, 1.0, 1.0])  # Assuming 3 classes, adjust weights as needed
        #class_weights = class_weights.to(model.device)

        # compute categorical cross-entropy loss
        loss_fct = nn.CrossEntropyLoss().to(model.device)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

In [ ]:
training_args = TrainingArguments(output_dir = './data/',
                                  num_train_epochs = 5,
                                  per_device_train_batch_size = 16,
                                  per_device_eval_batch_size = 16,
                                  logging_dir = './data/log/',
                                  seed = 1031,
                                  learning_rate=5e-06,
                                  weight_decay=0.01,
                                  save_strategy='steps',
                                  evaluation_strategy='steps',
                                  save_steps = 1000,
                                  eval_steps = 1000,
                                  #lr_scheduler_type = 'cosine',
                                  load_best_model_at_end=True
                                  )

In [ ]:
trainer = CustomTrainer(model = model,
                  args = training_args,
                  train_dataset = train_dataset,
                  eval_dataset = valid_dataset,
                  compute_metrics = compute_metrics,
                  callbacks = [EarlyStoppingCallback(early_stopping_patience=5)],
                  data_collator=DataCollatorWithPadding(tokenizer=tokenizer))

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
trainer.train()

In [ ]:
trainer.evaluate(eval_dataset = test_dataset)

모델 불러오기

In [ ]:
import torch
import torch.nn.functional as F
from transformers import ElectraModel, ElectraTokenizer, ElectraForSequenceClassification

PATH = './model/'
device = torch.device('cuda')
model = torch.load(PATH + 'model3.pt')
model.load_state_dict(torch.load(PATH + 'model3_state_dict.pt'))
model.to(device)

추론(원하는 문장 직접 입력을 통한)

In [ ]:
import torch
import torch.nn.functional as F

def sentence_predict(sent):
    model.eval()

    # 모델과 토크나이저 로드 (미리 로드되어 있다고 가정)
    # model = ElectraForSequenceClassification.from_pretrained("kcelectra-model-name")
    # tokenizer = ElectraTokenizer.from_pretrained("kcelectra-tokenizer-name")

    tokenized_sent = tokenizer(sent,
                               return_tensors='pt',
                               truncation=True,
                               add_special_tokens=True,
                               padding=True,
                               max_length=128)

    tokenized_sent.to(device)

    with torch.no_grad():
        outputs = model(input_ids=tokenized_sent["input_ids"],
                        attention_mask=tokenized_sent["attention_mask"],
                        token_type_ids=tokenized_sent["token_type_ids"]
                        )

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()[0]
    
    labels = ["성적인 발언", "정상 발언", "혐오 발언"]

    result_str = f"입력 문장: {sent}\n"
    
    for i, logit in enumerate(logits):
        if i == 0 or i == 1 or i == 2:
            logit = max(0, logit)  # 음수인 경우 0으로 바꾸기
        #result_str += f"Class {i} Logit: {logit:.4f}\n"
    
    predicted_class = logits.argmax()
    result_str += f"예측 결과: {labels[predicted_class]}"

    # 각 클래스에 대한 백분율 계산
    total_sum = sum([max(0, logit) for logit in logits[:3]])  # 음수인 경우 0으로 바꾼 값의 합
    percentages = [(max(0, logit) / total_sum) * 100 for logit in logits[:3]]
    for i, percent in enumerate(percentages):
        result_str += f"\nClass {i}: {percent:.2f}%"

    return result_str



while True:
    sentence = input("문장을 입력하세요 : ")
    if sentence == '0':
        break

    print(sentence_predict(sentence))
    print("\n")

그루밍 대화 데이터 전처리

In [ ]:
import os
import pandas as pd
import re

# 불용어 파일 경로
stopwords_file_path = './stopwords.txt' 

# 불용어 파일 읽기
with open(stopwords_file_path, 'r', encoding='utf-8') as stopwords_file:
    stopwords = stopwords_file.read().splitlines()

# 불용어 제거 함수
def remove_rows_with_stopwords(df, stopwords):
    df['불용어 포함'] = df['문장'].apply(lambda x: any(word in x for word in stopwords))
    return df[~df['불용어 포함']].drop(columns=['불용어 포함'])

# 'ㅋ'나 'ㅎ'만 있는 행 제거 함수
def remove_rows_with_korean_emojis(df):
    pattern = re.compile(r'^[ㅋㅎㅠ]+$')  # 'ㅋ'나 'ㅎ'만 있는 패턴
    url_pattern = re.compile(r'http[s]?://\S+')  # URL 패턴
    df['삭제할 행'] = df['문장'].apply(lambda x: True if pattern.match(x) or url_pattern.match(x) else False)
    return df[~df['삭제할 행']].drop(columns=['삭제할 행'])

# 디렉토리 경로 설정
directory_path = './data/talk/'  # txt 파일들이 들어있는 디렉토리 경로
directory_output_path = './data/talk/preprocess' # 전처리 된 파일을 저장할 경로

# 모든 txt 파일의 파일명을 가져옴
file_names = [file for file in os.listdir(directory_path) if file.endswith('.txt')]

# 모든 파일을 읽어와서 전처리 후 다시 파일로 저장
for file_name in file_names:
    file_path = os.path.join(directory_path, file_name)
    with open(file_path, 'r', encoding='utf-8') as file:
        data_list = file.readlines()

    # 데이터 프레임 생성
    column_name = '문장'
    df = pd.DataFrame(data_list, columns=[column_name])

    # 불용어 제거 적용
    df = remove_rows_with_stopwords(df, stopwords)

    # 'ㅋ'나 'ㅎ'만 있는 행 제거 적용
    df = remove_rows_with_korean_emojis(df)

    # 문장 길이가 4보다 큰 행만 남기기
    df = df[df['문장'].apply(lambda x: len(x) > 4)].reset_index(drop=True)

    # \n 제거
    df['문장'] = df['문장'].str.replace('\n', '')

    # 전처리된 데이터를 파일로 저장
    output_file_path = os.path.join(directory_output_path, f'preprocessed_{file_name}')
    df.to_csv(output_file_path, index=False, encoding='utf-8')


전처리 된 txt 파일을 통한 추론 후 csv 파일로 저장

In [ ]:
import os
import pandas as pd

def sentence_predict(sent):
    model.eval()

    tokenized_sent = tokenizer(sent,
                               return_tensors = 'pt',
                               truncation = True,
                               add_special_tokens = True,
                               padding = True,
                               max_length = 128)
    
    tokenized_sent.to(device)

    with torch.no_grad():
        outputs = model(input_ids = tokenized_sent["input_ids"],
                        attention_mask = tokenized_sent["attention_mask"],
                        token_type_ids = tokenized_sent["token_type_ids"]
                        )
    
    logits = outputs[0]
    logits = logits.detach().cpu()
    result = logits.argmax(-1)

    if result == 0:
        result =" >> 성적인 발언"
    elif result == 1:
        result = " >> 정상 발언"
    elif result == 2:
        result = " >> 혐오 발언"
    return result

# 디렉토리 경로 설정
directory_path = './data/talk/preprocess/'  # txt 파일들이 들어있는 디렉토리 경로

# 디렉토리 내의 모든 txt 파일에 대해 처리
for filename in os.listdir(directory_path):
    if filename.endswith('.txt'):
        txt_file_path = os.path.join(directory_path, filename)

        # 결과를 저장할 데이터 프레임 생성
        results_df = pd.DataFrame(columns=['문장', '결과'])

        with open(txt_file_path, 'r', encoding='utf-8') as f:
            for line in f:
                sentence = line.strip()  # 개행 문자 제거
                if sentence == '0':
                    break

                result = sentence_predict(sentence)
                tokenized_sent = tokenizer(sentence,
                                           return_tensors='pt',
                                           truncation=True,
                                           add_special_tokens=True,
                                           padding=True,
                                           max_length=128)
                tokenized_sent.to(device)

                with torch.no_grad():
                    outputs = model(input_ids=tokenized_sent["input_ids"],
                                    attention_mask=tokenized_sent["attention_mask"],
                                    token_type_ids=tokenized_sent["token_type_ids"]
                                    )
                logits = outputs[0]
                logits = logits.detach().cpu().numpy()[0]
                labels = ["성적인 발언", "정상 발언", "혐오 발언"]

                result_str = f"입력 문장: {sentence}\n"
    
                for i, logit in enumerate(logits):
                    if i == 0 or i == 1 or i == 2:
                        logit = max(0, logit)  # 음수인 경우 0으로 바꾸기
        
                predicted_class = logits.argmax()
                result_str += f"예측 결과: {labels[predicted_class]}"

                # 각 클래스에 대한 백분율 계산
                total_sum = sum([max(0, logit) for logit in logits[:3]])  # 음수인 경우 0으로 바꾼 값의 합
                percentages = [(max(0, logit) / total_sum) * 100 for logit in logits[:3]]
                for i, percent in enumerate(percentages):
                    result_str += f"\nClass {i}: {percent:.2f}%"
                

                # 결과 및 소프트맥스 함수를 적용하기 전 로짓 값을 데이터 프레임에 추가
                results_df = results_df.append({
                    '문장': sentence,
                    '결과': result,
                    'Class 0': percentages[0],
                    'Class 1': percentages[1],
                    'Class 2': percentages[2]
                }, ignore_index=True)
        
        # 결과를 CSV 파일로 저장
        output_file_path = os.path.join(directory_path, f'results_{filename}.csv')
        results_df.to_csv(output_file_path, index=False, encoding='cp949')

txt 파일 안에 문장을 읽어와 리스트 변경 후 결과값 리스트 형으로 출력

In [ ]:
def sentence_predict(sent):

    model.eval()

    MODEL_NAME = "beomi/KcELECTRA-base-v2022"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    tokenized_sent = tokenizer(sent,
                               return_tensors='pt',
                               truncation=True,
                               add_special_tokens=True,
                               padding=True,
                               max_length=128)

    tokenized_sent.to(device)

    with torch.no_grad():
        outputs = model(input_ids=tokenized_sent["input_ids"],
                        attention_mask=tokenized_sent["attention_mask"],
                        token_type_ids=tokenized_sent["token_type_ids"]
                        )

    logits = outputs[0]
    logits = logits.detach().cpu().numpy()[0]
    
    labels = ["성적인 발언", "정상 발언", "혐오 발언"]
    result_str = f"입력 문장: {sent}\n"
    
    for i, logit in enumerate(logits):
        if i == 0 or i == 1 or i == 2:
            logit = max(0, logit)  # 음수인 경우 0으로 바꾸기
        #result_str += f"Class {i} Logit: {logit:.4f}\n"
    
    predicted_class = logits.argmax()
    result_str += f"예측 결과: {labels[predicted_class]}"

    # 각 클래스에 대한 백분율 계산
    total_sum = sum([max(0, logit) for logit in logits[:3]])  # 음수인 경우 0으로 바꾼 값의 합
    percentages = [(max(0, logit) / total_sum) * 100 for logit in logits[:3]]
    for i, percent in enumerate(percentages):
        result_str += f"\nClass {i}: {percent:.2f}%"

    return result_str

input_folder = "C:/Users/choiyunjin/Desktop/test/talk/preprocess/" # 경로 지정
output_results = []

for filename in os.listdir(input_folder): # 경로 안에 txt 파일을 읽어와 리스트 형태로 변환한 뒤 추론
    if filename.endswith(".txt"):
        with open(os.path.join(input_folder, filename), "r", encoding="utf-8") as f:
            sentences = [line.strip() for line in f]
            results = [sentence_predict(sentence) for sentence in sentences]
            output_results.extend(results)

for result in output_results:
    print(result)
    print("\n")

print(output_results)

데이터 증강

In [ ]:
import random
import pickle

# 동의어 대체 함수를 활용하여 문장 증강
def synonym_replacement(words, wordnet_dict, n):
    new_words = words.copy()
    random_word_list = list(set([word for word in words]))
    random.shuffle(random_word_list)
    num_replaced = 0
    for random_word in random_word_list:
        synonyms = get_synonyms(random_word, wordnet_dict)
        if len(synonyms) >= 1:
            synonym = random.choice(list(synonyms))
            new_words = [synonym if word == random_word else word for word in new_words]
            num_replaced += 1
        if num_replaced >= n:
            break

    return new_words

def get_synonyms(word, wordnet_dict):
    synonyms = []
    if word in wordnet_dict:
        synonyms = wordnet_dict[word]
    return synonyms

# 단어 위치 바꾸기 증강 함수
def word_order_augmentation(sentence):
    words = sentence.split()
    augmented_sentences = []

    for _ in range(len(words)):
        random.shuffle(words)
        augmented_sentences.append(" ".join(words))

    return augmented_sentences

# 기존 동의어 사전 pickle 파일 로드
with open("wordnet.pickle", "rb") as f:
    wordnet_dict = pickle.load(f)

# 텍스트 파일에서 문장들을 읽어와 리스트로 저장
with open("extra_data.txt", "r", encoding="utf-8") as f:
    sentences = f.readlines()

# 동의어 대체와 단어 위치 바꾸기 증강을 수행한 문장들을 저장할 리스트
augmented_sentences = []

for sentence in sentences:
    sentence = sentence.strip()  # 줄바꿈 문자 제거
    # 동의어 대체 증강을 3번 수행
    for _ in range(3):
        augmented_words = synonym_replacement(sentence.split(), wordnet_dict, n=1)
        augmented_sentences.append(" ".join(augmented_words))
    # 단어 위치 바꾸기 증강
    augmented_sentences.extend(word_order_augmentation(sentence))

# 증강된 문장들을 파일에 저장
with open("augmented_sentences.txt", "w", encoding="utf-8") as f:
    for sentence in augmented_sentences:
        f.write(sentence + "\n")